In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
Q1: Spark version

In [ ]:
spark.version

In [ ]:
import pyarrow.parquet as pq

file_path = "yellow_tripdata_2024-10.parquet"
parquet_file = pq.ParquetFile(file_path)

print(parquet_file.schema)

In [ ]:
from pyspark.sql import types

schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True),
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True),
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True),
    types.StructField('passenger_count', types.LongType(), True),
    types.StructField('trip_distance', types.DoubleType(), True),
    types.StructField('RatecodeID', types.LongType(), True),
    types.StructField('store_and_fwd_flag', types.StringType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('payment_type', types.LongType(), True),
    types.StructField('fare_amount', types.DoubleType(), True),
    types.StructField('extra', types.DoubleType(), True),
    types.StructField('mta_tax', types.DoubleType(), True),
    types.StructField('tip_amount', types.DoubleType(), True),
    types.StructField('tolls_amount', types.DoubleType(), True),
    types.StructField('improvement_surcharge', types.DoubleType(), True),
    types.StructField('total_amount', types.DoubleType(), True),
    types.StructField('congestion_surcharge', types.DoubleType(), True),
    types.StructField('Airport_fee', types.DoubleType(), True)
])

In [ ]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .parquet('yellow_tripdata_2024-10.parquet')

df.printSchema()

df = df.repartition(4)

df.write.parquet('data/pq/hw/2024/10/')

In [ ]:
df = spark.read.parquet('data/pq/hw/2024/10/')

In [ ]:
Q2: What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? 

In [ ]:
import os
import glob

parquet_dir = "data/pq/hw/2024/10/"
parquet_files = glob.glob(os.path.join(parquet_dir, "*.parquet"))

total_size_bytes = sum(os.path.getsize(file) for file in parquet_files)
total_size_mb = total_size_bytes / (1024 * 1024)

average_size_mb = total_size_mb / len(parquet_files) if parquet_files else 0

print(f"Total files: {len(parquet_files)}")
print(f"Total size: {total_size_mb:.2f} MB")
print(f"Average file size: {average_size_mb:.2f} MB")

# Or in Terminal ../code:
# awk '{s+=$1; n++} END {print "Average size (MB):", s/n/1024/1024}' < <(find data/pq/hw/2024/10/ -type f -name "*.parquet" -printf "%s\n")

In [ ]:
Q3: How many taxi trips were there on the 15th of October? Consider only trips that started on the 15th of October.

In [ ]:
from pyspark.sql import functions as F

In [ ]:
trips_oct15 = df.filter(F.to_date(df.tpep_pickup_datetime) == "2024-10-15")
trip_count = trips_oct15.count()
print(f"Number of trips on October 15th: {trip_count}")

In [ ]:
Q4: What is the length of the longest trip in the dataset in hours?

In [ ]:
from pyspark.sql import functions as F

df = spark.read.parquet("data/pq/hw/2024/10/")
df = df.withColumn("trip_duration_hours", 
                   (F.unix_timestamp(df.tpep_dropoff_datetime) - F.unix_timestamp(df.tpep_pickup_datetime)) / 3600)
max_duration = df.select(F.max("trip_duration_hours")).collect()[0][0]

print(f"Longest trip duration: {max_duration:.2f} hours")

In [ ]:
Q6: Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

In [ ]:
from pyspark.sql import functions as F

zone_df = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")
zone_df.createOrReplaceTempView("zones")

df = spark.read.parquet("data/pq/hw/2024/10/")
df.createOrReplaceTempView("yellow_taxi")

result = spark.sql("""
    SELECT z.Zone, COUNT(y.PULocationID) AS trip_count
    FROM yellow_taxi y
    JOIN zones z ON y.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY trip_count ASC
    LIMIT 1
""")

result.show()